# Clase 3 — SQL para la Exploración de Datos (Data Discovery)
**Módulo 1 · Unidad 1 · DevSeniorCode**

---

## El caso de negocio

**DataJobs** es una startup colombiana que conecta empresas tech con talento especializado.
Llevamos un año recolectando datos: candidatos registrados, ofertas publicadas y postulaciones realizadas.
La directora de operaciones, Camila Reyes, quiere respuestas:

> *¿Tenemos buen talento? ¿Las empresas están encontrando lo que buscan? ¿Por qué algunas ofertas llevan meses abiertas sin llenarse?*

Esas preguntas las respondemos hoy, con SQL.

---

## BLOQUE 1 — Configuración del entorno

**¿Qué es SQLite?**

SQLite es un motor de base de datos que ya viene instalado con Python. No necesita servidor ni configuración — 
crea la base de datos como un archivo en memoria RAM. Es el mismo lenguaje SQL estándar que usa PostgreSQL, MySQL y BigQuery.

**Flujo de hoy:**
```
CSV (candidatos, ofertas, postulaciones)
        ↓
   Pandas (lector de CSV)
        ↓
   SQLite (motor de base de datos en memoria)
        ↓
   SQL PURO — preguntas a los datos
        ↓
   (Clase 4) → pandas.read_sql() → DataFrame → Modelo de IA
```

In [ ]:
# ─── CELDA 1: Importaciones ─────────────────────────────────────────────────
# sqlite3: módulo incluido en Python. Permite crear y consultar bases de datos SQLite.
# pandas: para cargar los CSV y mostrar resultados de SQL como tablas limpias.
# pathlib: para construir rutas de archivos de forma segura en cualquier sistema operativo.

# Escriban las tres importaciones:
import sqlite3
import pandas as pd
from pathlib import Path


In [2]:
# ─── CELDA 2: Crear la conexión y el cursor ─────────────────────────────────
# sqlite3.connect(':memory:') crea una base de datos en la RAM del computador.
# ':memory:' es un nombre especial de SQLite — no crea ningún archivo en el disco.
# conn es el objeto que representa la conexión abierta con la base de datos.
# cursor es el objeto que ejecuta los comandos SQL dentro de esa conexión.

# Creen la conexión y el cursor:
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

print('✅ Conexión a SQLite establecida')
print(f'Versión de SQLite: {sqlite3.sqlite_version}')


✅ Conexión a SQLite establecida
Versión de SQLite: 3.51.2


In [4]:
# ─── CELDA 3: Cargar los CSVs a SQLite ──────────────────────────────────────
# Paso 1: pd.read_csv() lee cada archivo CSV y lo convierte en un DataFrame.
# Paso 2: .to_sql() escribe ese DataFrame como tabla dentro de SQLite.
# if_exists='replace': si la tabla ya existe, la reemplaza (útil al re-ejecutar).
# index=False: no guarda el índice interno de Pandas como columna extra.

# Carguen los tres CSV (nombres: candidatos.csv, ofertas.csv, postulaciones.csv)
# y créenlos como tablas con to_sql:
df_candidatos = pd.read_csv('candidatos.csv')
df_ofertas = pd.read_csv('ofertas.csv')
df_postulaciones = pd.read_csv('postulaciones.csv')

# Pasamos a tablas
df_candidatos.to_sql('candidatos', conn, if_exists='replace', index=False)
df_ofertas.to_sql('ofertas', conn, if_exists='replace', index=False)
df_postulaciones.to_sql('postulaciones', conn, if_exists='replace', index=False)

print(f'✅ candidatos    → {len(df_candidatos)} filas')
print(f'✅ ofertas       → {len(df_ofertas)} filas')
print(f'✅ postulaciones → {len(df_postulaciones)} filas')


✅ candidatos    → 1200 filas
✅ ofertas       → 400 filas
✅ postulaciones → 2000 filas


In [10]:
# ─── CELDA 4: Función auxiliar ───────────────────────────────────────────────
# Esta función la usaremos toda la clase para ejecutar SQL y ver resultados bonitos.
# pd.read_sql_query() ejecuta el query SQL en la conexión y devuelve un DataFrame.
# query: el string con el código SQL.
# conn: la conexión a SQLite que creamos arriba.

def ejecutar_sql(query):
    # Escriban el cuerpo de la función (una sola línea con return y pd.read_sql_query):
    return pd.read_sql_query(query, conn)    

print('✅ Función ejecutar_sql() lista')


✅ Función ejecutar_sql() lista


---

## BLOQUE 2 — SELECT, DISTINCT y Alias

### SELECT — La pregunta más básica

**Definición técnica (ISO/IEC 9075):**
`SELECT` es la cláusula DQL (Data Query Language) que especifica qué columnas se desean recuperar de una o más tablas.

**Anatomía:**
```sql
SELECT  columna1, columna2   -- ¿Qué quiero ver?
FROM    nombre_tabla          -- ¿De dónde?
LIMIT   N                     -- ¿Cuántas filas máximo?
```

**Analogía:** SELECT es el camarero que trae exactamente lo que pediste — ni más, ni menos.

In [11]:
# ─── SELECT básico con LIMIT ─────────────────────────────────────────────────
# El asterisco (*) es comodín: significa 'todas las columnas'.
# LIMIT 5: muestra solo las primeras 5 filas — útil para explorar sin saturar la pantalla.

# Escriban el query: seleccionen todo de la tabla candidatos, límite 5 filas
query_01 = """
SELECT * 
FROM candidatos 
LIMIT 5
"""
ejecutar_sql(query_01)


,candidato_id,nombre,ciudad,area_interes,nivel,anos_experiencia,skills_principales,salario_esperado_cop,disponible,fecha_registro
0,1,Natalia López,Montería,Data Science,Senior,6,TensorFlow|Scikit-learn|Python|SQL,7785972,Sí,2024-09-15
1,2,Alejandro García,Montería,Cloud,Semi-Senior,5,GCP|AWS|Azure|Kubernetes,5165476,Sí,2024-02-22
2,3,Camila Moreno,Pasto,Backend,Senior,5,Python|SQL|Spring Boot|Node.js,7583470,Sí,2024-05-28
3,4,Camila Molina,Pereira,DevOps,Junior,1,Jenkins|Linux,3405459,Sí,2024-09-30
4,5,Miguel Ramírez,Ibagué,Frontend,Lead,11,Vue.js|JavaScript|React|CSS,11769852,Sí,2024-04-18


In [13]:
# ─── SELECT de columnas específicas ─────────────────────────────────────────
# Camila quiere ver SOLO nombre y ciudad de los candidatos.
# En lugar de *, listamos las columnas que queremos separadas por coma.

# Escriban el query: seleccionen nombre y ciudad, límite 10
query_02 = """
SELECT nombre, ciudad 
FROM candidatos 
LIMIT 10

"""

ejecutar_sql(query_02)


,nombre,ciudad
0,Natalia López,Montería
1,Alejandro García,Montería
2,Camila Moreno,Pasto
3,Camila Molina,Pereira
4,Miguel Ramírez,Ibagué
5,Natalia Herrera,Montería
6,Paula Herrera,Villavicencio
7,Miguel López,Santa Marta
8,Daniel Ramos,Pasto
9,Camila Martínez,Tunja


### DISTINCT — Eliminar duplicados

**Definición técnica:** `DISTINCT` elimina filas duplicadas del resultado del SELECT, retornando únicamente valores únicos.

**Analogía:** Toman una lista de 1.200 ciudades y tachan las repetidas. DISTINCT hace eso en milisegundos.

**Regla:** `SELECT DISTINCT columna FROM tabla`

In [ ]:
# ─── DISTINCT: ciudades únicas ───────────────────────────────────────────────
# Sin DISTINCT: Bogotá aparecería ~300 veces.
# Con DISTINCT: Bogotá aparece una sola vez.
# ORDER BY: ordena el resultado. ASC = ascendente (A→Z), DESC = descendente (Z→A).

# Escriban: ciudades únicas de candidatos, ordenadas alfabéticamente
query_03 = """
SELECT DISTINCT ciudad
FROM candidatos
ORDER BY ciudad ASC
"""

ejecutar_sql(query_03)


,ciudad
0,Armenia
1,Barranquilla
2,Bogotá
3,Bucaramanga
4,Cali
5,Cartagena
6,Cúcuta
7,Ibagué
8,Manizales
9,Medellín


In [17]:
# ─── Nivel 2: DISTINCT áreas de interés ─────────────────────────────────────
# Camila quiere saber qué áreas de interés tienen los candidatos registrados.
# Usen DISTINCT para que cada área aparezca una sola vez.

query_04 = """
SELECT DISTINCT area_interes
FROM candidatos
ORDER BY area_interes ASC

"""

ejecutar_sql(query_04)


,area_interes
0,BI & Analytics
1,Backend
2,Ciberseguridad
3,Cloud
4,Data Science
5,DevOps
6,Diseño UX/UI
7,Frontend
8,Full Stack
9,Machine Learning


### Alias — Nombres legibles en los resultados

**Definición técnica:** Un alias es un nombre temporal asignado a una columna mediante la palabra clave `AS`. No modifica la tabla — solo cambia cómo aparece en el resultado del query.

**Cuándo usarlo:** Cuando el resultado va a un reporte o cuando el nombre de la columna es técnico y poco legible.

In [29]:
# ─── ALIAS: renombrar columnas en el resultado ───────────────────────────────
# AS le da un nombre más legible a la columna en el resultado.
# El alias solo existe dentro de este query — no cambia la tabla original.
# Nombres con espacios van entre comillas dobles: AS "Nombre Completo"

# Seleccionen: nombre (alias 'Candidato'), ciudad, nivel (alias 'Nivel Profesional')
# y salario_esperado_cop (alias 'Salario Esperado (COP)'). Límite 8 filas.
query_05 = """
SELECT 
    nombre AS "Candidato", 
    ciudad, 
    nivel AS "Nivel Profesional", 
    salario_esperado_cop AS "Salario Esperado (COP)"
FROM candidatos
LIMIT 8
"""

ejecutar_sql(query_05)


,Candidato,ciudad,Nivel Profesional,Salario Esperado (COP)
0,Natalia López,Montería,Senior,7785972
1,Alejandro García,Montería,Semi-Senior,5165476
2,Camila Moreno,Pasto,Senior,7583470
3,Camila Molina,Pereira,Junior,3405459
4,Miguel Ramírez,Ibagué,Lead,11769852
5,Natalia Herrera,Montería,Lead,13036646
6,Paula Herrera,Villavicencio,Junior,3249668
7,Miguel López,Santa Marta,Senior,11258742


---

## BLOQUE 3 — WHERE, IN, BETWEEN y Operadores Lógicos

### WHERE — El filtro fundamental

**Definición técnica:** `WHERE` filtra filas según una condición lógica. Solo las filas que la cumplen aparecen en el resultado. Se evalúa ANTES de SELECT — primero filtra, luego proyecta.

**Analogía:** WHERE es el portero del club. SELECT decide qué información mostrar de los que entran; WHERE decide quién entra.

**Orden obligatorio en SQL:**
```
SELECT → FROM → WHERE → GROUP BY → HAVING → ORDER BY → LIMIT
```

In [ ]:
# ─── WHERE con AND: candidatos de Bogotá disponibles ────────────────────────
# AND exige que AMBAS condiciones se cumplan simultáneamente.
# Texto en SQL: siempre entre comillas SIMPLES ' ', no dobles.
# El signo = compara exactamente con el valor indicado.

# Seleccionen nombre, ciudad, area_interes, disponible
# WHERE ciudad sea Bogotá AND disponible sea 'Sí'
# Límite 10
query_06 = """
SELECT 
    nombre, 
    ciudad, 
    area_interes, 
    disponible
FROM candidatos
WHERE ciudad = 'Bogotá' AND disponible = 'Sí'
LIMIT 10
"""

ejecutar_sql(query_06)


,nombre,ciudad,area_interes,disponible
0,Cristian Mendoza,Bogotá,QA,Sí
1,Camila Ortiz,Bogotá,Frontend,Sí
2,Sebastián Jiménez,Bogotá,Ciberseguridad,Sí
3,Juliana Ramírez,Bogotá,Cloud,Sí
4,Natalia Pérez,Bogotá,Frontend,Sí
5,Javier Vargas,Bogotá,Frontend,Sí
6,Santiago Pérez,Bogotá,Machine Learning,Sí
7,Luisa Pérez,Bogotá,Ciberseguridad,Sí
8,Andrés Flores,Bogotá,Mobile,Sí
9,David Guerrero,Bogotá,QA,Sí


In [31]:
# ─── Nivel 2: WHERE con OR y paréntesis ─────────────────────────────────────
# OR acepta que CUALQUIERA de las condiciones se cumpla.
# Paréntesis: agrupan lógica, igual que en matemáticas. Sin ellos, AND tiene precedencia.

# Candidatos Senior o Lead de Medellín
# Columnas: nombre, ciudad, nivel, anos_experiencia
query_07 = """
SELECT 
    nombre, 
    ciudad, 
    nivel, 
    anos_experiencia
FROM candidatos
WHERE (nivel = 'Senior' OR nivel = 'Lead') AND ciudad = 'Medellín'

"""

ejecutar_sql(query_07)


,nombre,ciudad,nivel,anos_experiencia
0,Nicolás Reyes,Medellín,Lead,11
1,Felipe Reyes,Medellín,Lead,13
2,Diego Suárez,Medellín,Lead,11
3,Sara Mendoza,Medellín,Lead,10
4,Carolina Mendoza,Medellín,Senior,8
5,Mateo Castro,Medellín,Senior,9
6,Juliana Moreno,Medellín,Lead,8
7,Sara Molina,Medellín,Lead,15
8,Mateo García,Medellín,Lead,10
9,Carolina Moreno,Medellín,Lead,11


### IN — Filtrar múltiples valores

**Definición técnica:** `IN` comprueba si un valor pertenece a un conjunto listado. Es equivalente a múltiples `OR` pero más legible y eficiente.

```sql
WHERE ciudad IN ('Bogotá', 'Medellín', 'Cali')
-- equivale a:
WHERE ciudad = 'Bogotá' OR ciudad = 'Medellín' OR ciudad = 'Cali'
```

In [35]:
# ─── IN: candidatos de tres ciudades disponibles ─────────────────────────────
# IN recibe una lista de valores entre paréntesis, separados por comas.
# Más legible que múltiples OR y menos propenso a errores de tipeo.

# Candidatos disponibles de Bogotá, Medellín o Cali
# Columnas: nombre, ciudad, area_interes, nivel
# Ordenados por ciudad, límite 15
query_08 = """
SELECT 
    nombre, 
    ciudad, 
    area_interes, 
    nivel
FROM candidatos
WHERE ciudad IN ('Bogotá', 'Medellín', 'Cali') AND disponible = 'Sí'
ORDER BY ciudad ASC
LIMIT 15
"""

ejecutar_sql(query_08)


,nombre,ciudad,area_interes,nivel
0,Cristian Mendoza,Bogotá,QA,Lead
1,Camila Ortiz,Bogotá,Frontend,Senior
2,Sebastián Jiménez,Bogotá,Ciberseguridad,Lead
3,Juliana Ramírez,Bogotá,Cloud,Semi-Senior
4,Natalia Pérez,Bogotá,Frontend,Lead
5,Javier Vargas,Bogotá,Frontend,Lead
6,Santiago Pérez,Bogotá,Machine Learning,Semi-Senior
7,Luisa Pérez,Bogotá,Ciberseguridad,Junior
8,Andrés Flores,Bogotá,Mobile,Semi-Senior
9,David Guerrero,Bogotá,QA,Senior


### BETWEEN — Rangos de valores

**Definición técnica:** `BETWEEN` filtra valores dentro de un rango **inclusivo**. `BETWEEN a AND b` equivale a `>= a AND <= b`. Funciona con números, fechas y texto.

In [36]:
# ─── BETWEEN: rango de experiencia ──────────────────────────────────────────
# BETWEEN es INCLUSIVO: incluye el 3 y el 6.
# DESC en ORDER BY: ordena de mayor a menor (los más experimentados primero).

# Candidatos con entre 3 y 6 años de experiencia
# Columnas: nombre, nivel, anos_experiencia, area_interes
# Ordenados por experiencia descendente, límite 15
query_09 = """
SELECT 
    nombre, 
    nivel, 
    anos_experiencia, 
    area_interes
FROM candidatos
WHERE anos_experiencia BETWEEN 3 AND 6
ORDER BY anos_experiencia DESC
LIMIT 15
"""

ejecutar_sql(query_09)


,nombre,nivel,anos_experiencia,area_interes
0,Natalia López,Senior,6,Data Science
1,Laura Rodríguez,Senior,6,Mobile
2,Mateo Castro,Senior,6,BI & Analytics
3,María González,Senior,6,Cloud
4,Paula Sánchez,Senior,6,QA
5,Laura López,Senior,6,Data Science
6,Santiago González,Senior,6,Diseño UX/UI
7,Sara Romero,Senior,6,QA
8,Mariana Ríos,Senior,6,Diseño UX/UI
9,Javier Martínez,Senior,6,Full Stack


In [37]:
# ─── Nivel 3: Desde cero — ofertas activas remotas en Q1 2024 ───────────────
# Camila quiere ofertas publicadas entre el 01/01/2024 y el 31/03/2024
# que estén activas y sean remotas.
# Columnas: empresa, area, nivel_requerido, modalidad, fecha_publicacion
# Pista: BETWEEN funciona con fechas en formato 'YYYY-MM-DD'

query_10 = """
SELECT 
    empresa, 
    area, 
    nivel_requerido, 
    modalidad, 
    fecha_publicacion
FROM ofertas
WHERE fecha_publicacion BETWEEN '2024-01-01' AND '2024-03-31'
    AND estado = 'Activa'
    AND modalidad = 'Remoto'
"""

ejecutar_sql(query_10)


,empresa,area,nivel_requerido,modalidad,fecha_publicacion
0,Pragma,BI & Analytics,Semi-Senior,Remoto,2024-01-29
1,MercadoLibre,Full Stack,Lead,Remoto,2024-03-01
2,Platzi,Backend,Junior,Remoto,2024-03-02
3,Truora,Frontend,Junior,Remoto,2024-03-22
4,TechCorp Colombia,Frontend,Semi-Senior,Remoto,2024-02-25
5,Platzi,Diseño UX/UI,Junior,Remoto,2024-01-10
6,Konrad Lorenz Tech,Backend,Semi-Senior,Remoto,2024-03-24
7,Addi,Machine Learning,Lead,Remoto,2024-01-12
8,Runa,BI & Analytics,Senior,Remoto,2024-03-26
9,Pragma,Full Stack,Junior,Remoto,2024-03-22


---

---

## BLOQUE 4 — GROUP BY, HAVING y Funciones de Agregación

### Funciones de agregación

**Definición técnica:** Las funciones de agregación operan sobre un conjunto de filas y retornan un único valor resumen.

| Función | Qué hace |
|---------|----------|
| `COUNT(*)` | Cuenta todas las filas (incluyendo nulos) |
| `COUNT(col)` | Cuenta filas donde esa columna no es nula |
| `AVG(col)` | Promedio aritmético de valores numéricos |
| `SUM(col)` | Suma de todos los valores |
| `MAX(col)` | Valor más alto |
| `MIN(col)` | Valor más bajo |
| `ROUND(val, n)` | Redondea a n decimales |

**Analogía:** COUNT() cuenta 1.200 hojas de papel. SUM() suma un número de cada hoja. AVG() calcula el promedio. Lo que tomaría horas a mano, SQL lo hace en milisegundos.

In [39]:
# ─── COUNT: total de candidatos ──────────────────────────────────────────────
# COUNT(*) cuenta TODAS las filas, incluyendo las que tienen nulos.
# El alias AS le da un nombre al resultado en la tabla de salida.

# Cuántos candidatos hay en total
query_11 = """
SELECT COUNT(*) AS "Total de Candidatos"
FROM candidatos
"""

ejecutar_sql(query_11)


,Total de Candidatos
0,1200


In [40]:
# ─── AVG, MAX, MIN: estadísticas de salario ──────────────────────────────────
# AVG() calcula el promedio aritmético de una columna numérica.
# ROUND(valor, 0) redondea a 0 decimales — útil para salarios en pesos.
# MAX() y MIN() dan el rango de la distribución.

# Salario promedio (redondeado), experiencia promedio (1 decimal),
# salario máximo y mínimo de todos los candidatos
query_12 = """
SELECT
    ROUND(AVG(salario_esperado_cop), 0) AS salario_promedio_cop,
    ROUND(AVG(anos_experiencia), 1) AS experiencia_promedio_anos,
    MAX(salario_esperado_cop) AS salario_maximo_cop,
    MIN(salario_esperado_cop) AS salario_minimo_cop
FROM candidatos
"""

ejecutar_sql(query_12)


,salario_promedio_cop,experiencia_promedio_anos,salario_maximo_cop,salario_minimo_cop
0,8041641.0,5.9,17999255,2010362


### GROUP BY — Agrupar para descubrir patrones

**Definición técnica:** `GROUP BY` agrupa filas con el mismo valor en columnas especificadas y aplica funciones de agregación a cada grupo por separado.

**Analogía:** Organizan 1.200 hojas en montones por ciudad. Luego COUNT() cuenta cada montón.

In [52]:
# ─── GROUP BY + COUNT: candidatos por ciudad ────────────────────────────────
# GROUP BY ciudad: divide los 1.200 registros en grupos, uno por ciudad.
# COUNT(*): cuenta cuántos registros hay en cada grupo.
# ORDER BY total DESC: muestra las ciudades con más talento primero.

# Candidatos agrupados por ciudad, ordenados de más a menos
query_13 = """
SELECT
    ciudad,
    COUNT(*)    AS total_candidatos  
FROM candidatos
GROUP BY ciudad
ORDER BY total_candidatos DESC


"""

ejecutar_sql(query_13)


,ciudad,total_candidatos
0,Bucaramanga,85
1,Villavicencio,81
2,Neiva,81
3,Tunja,73
4,Pasto,70
5,Manizales,70
6,Cúcuta,70
7,Ibagué,69
8,Cartagena,69
9,Santa Marta,68


In [54]:
# ─── Nivel 2: GROUP BY — candidatos por área con salario promedio ───────────
# Camila quiere: cuántos candidatos hay por área y cuál es su salario promedio.

# Completen con GROUP BY y las funciones de agregación correctas
query_14 = """
SELECT 
    area_interes                AS area,
    COUNT(*) AS total_candidatos,
    ROUND(AVG(salario_esperado_cop), 0)               AS salario_promedio
FROM candidatos
GROUP BY area_interes
ORDER BY salario_promedio DESC
"""

ejecutar_sql(query_14)


,area,total_candidatos,salario_promedio
0,QA,95,8661733.0
1,BI & Analytics,114,8648197.0
2,Full Stack,97,8437852.0
3,Diseño UX/UI,111,8217495.0
4,Machine Learning,96,8149400.0
5,Backend,101,8141911.0
6,Cloud,105,8046644.0
7,DevOps,101,7849452.0
8,Data Science,101,7681053.0
9,Mobile,78,7655430.0


### HAVING — Filtrar grupos

**Definición técnica:** `HAVING` filtra grupos DESPUÉS de que `GROUP BY` los formó. A diferencia de `WHERE` que filtra filas individuales, `HAVING` filtra grupos completos basándose en funciones de agregación.

**Regla de oro:**
- `WHERE` → filtra **filas** (antes de agrupar)
- `HAVING` → filtra **grupos** (después de agrupar)

In [60]:
# ─── HAVING: solo áreas con más de 80 candidatos ────────────────────────────
# HAVING va DESPUÉS de GROUP BY — el orden importa.
# HAVING COUNT(*) > 80: solo mostramos grupos donde el conteo supera 80.
# No podríamos usar WHERE aquí porque el COUNT no existe aún cuando WHERE se evalúa.

# Áreas con más de 80 candidatos, con su salario promedio
query_15 = """
SELECT
    area_interes    AS  Nombre_Area,
    ROUND(AVG(salario_esperado_cop),1)  AS Salario_promedio,
    COUNT(*)    AS Total
FROM candidatos
GROUP BY area_interes
HAVING COUNT(*) > 80
ORDER BY Total DESC

"""

ejecutar_sql(query_15)


,Nombre_Area,Salario_promedio,Total
0,BI & Analytics,8648197.3,114
1,Diseño UX/UI,8217494.7,111
2,Cloud,8046643.6,105
3,Frontend,7526787.8,102
4,DevOps,7849451.8,101
5,Data Science,7681053.0,101
6,Backend,8141910.5,101
7,Ciberseguridad,7349355.5,99
8,Full Stack,8437852.5,97
9,Machine Learning,8149400.0,96


In [64]:
# ─── Nivel 3: Desde cero — empresas con más de 5 ofertas ───────────────────
# Camila quiere: ¿qué empresas tienen más de 5 ofertas publicadas?
# Tabla: ofertas
# Mostrar: empresa, total de ofertas

query_16 = """
SELECT
    empresa AS  Nombre_empresa,
    COUNT(*)    AS Total_ofertas
FROM ofertas
GROUP BY empresa
HAVING COUNT(*) > 5
ORDER BY Total_ofertas DESC
"""

ejecutar_sql(query_16)


,Nombre_empresa,Total_ofertas
0,Bancolombia Digital,29
1,Truora,26
2,Pragma,25
3,Bold,25
4,Habi,22
5,Addi,22
6,Nequi,21
7,MercadoLibre,21
8,Globant,21
9,TechCorp Colombia,20


---

## BLOQUE 5 — JOINs: Cruzar Información de Negocio

**El problema:** La pregunta más importante de Camila requiere información de DOS tablas a la vez: '¿Quién se postuló a qué oferta?' — los candidatos están en una tabla y las ofertas en otra.

**Definición técnica:** Un `JOIN` combina filas de dos tablas basándose en una condición de relación entre columnas — generalmente una llave primaria (PK) y una llave foránea (FK).

**Conceptos clave:**
- **Primary Key (PK):** ID único de cada fila en su tabla. Ej: `candidato_id` en `candidatos`.
- **Foreign Key (FK):** Ese mismo ID referenciado en otra tabla. Ej: `candidato_id` en `postulaciones`.

**Analogía:** BUSCARV en Excel — buscan el ID en una lista y traen el nombre. Pero SQL lo hace para miles de filas en milisegundos.

### INNER JOIN — Solo los que coinciden

**Definición técnica:** `INNER JOIN` retorna únicamente las filas donde existe coincidencia en AMBAS tablas. Filas sin coincidencia son excluidas.

In [43]:
# ─── INNER JOIN: postulaciones + candidatos ─────────────────────────────────
# JOIN conecta dos tablas usando la columna que tienen en común: candidato_id.
# ON especifica la condición de unión: el puente entre tablas.
# Alias de tabla: p = postulaciones, c = candidatos.
# p.columna → columna de la tabla postulaciones.
# c.columna → columna de la tabla candidatos.

# Seleccionen: postulacion_id, nombre del candidato (alias 'candidato'),
# ciudad, area_interes, estado_postulacion, fecha_postulacion
# FROM postulaciones (alias p) INNER JOIN candidatos (alias c)
# ON la columna candidato_id de ambas tablas
# Límite 10
query_17 = """
SELECT
    p.postulacion_id,
    c.nombre                AS candidato,
    c.ciudad,
    c.area_interes,
    p.estado_postulacion,
    p.fecha_postulacion
FROM postulaciones p
INNER JOIN candidatos c ON p.candidato_id = c.candidato_id
LIMIT 10
"""

ejecutar_sql(query_17)


,postulacion_id,candidato,ciudad,area_interes,estado_postulacion,fecha_postulacion
0,1,Diego Ríos,Manizales,Ciberseguridad,Entrevista,2024-05-29
1,2,Nicolás Reyes,Cartagena,Data Science,Entrevista,2024-09-11
2,3,Sebastián Romero,Santa Marta,Backend,Contratada,2024-02-24
3,4,Javier Ruiz,Armenia,Frontend,En revisión,2024-07-17
4,5,Alejandro Reyes,Bucaramanga,Data Science,En revisión,2024-10-23
5,6,Natalia Castro,Tunja,Mobile,En revisión,2024-09-16
6,7,Gabriela Jiménez,Barranquilla,Machine Learning,En revisión,2024-03-23
7,8,Daniela García,Ibagué,BI & Analytics,En revisión,2024-07-24
8,9,Gabriela Romero,Ibagué,Diseño UX/UI,Rechazada,2024-07-12
9,10,Felipe Vega,Barranquilla,Cloud,Enviada,2024-03-19


In [ ]:
# ─── TRIPLE JOIN: candidatos + postulaciones + ofertas ───────────────────────
# Primer JOIN: postulaciones ↔ candidatos (por candidato_id)
# Segundo JOIN: resultado anterior ↔ ofertas (por oferta_id)
# Así funciona en empresas reales: datos distribuidos en varias tablas.

# Seleccionen: nombre del candidato, ciudad del candidato, empresa,
# area de la oferta, nivel_requerido, estado_postulacion, dias_hasta_respuesta
# Usando las tres tablas con sus dos JOINs
# Límite 15
query_18 = """

SELECT
    c.nombre              AS candidato,
    c.ciudad              AS ciudad_candidato,
    o.empresa,
    o.area,
    o.nivel_requerido,
    p.estado_postulacion,
    p.dias_hasta_respuesta
FROM postulaciones p
INNER JOIN candidatos c ON p.candidato_id = c.candidato_id
INNER JOIN ofertas    o ON p.oferta_id    = o.oferta_id
LIMIT 15
"""

ejecutar_sql(query_18)


In [45]:
# ─── Nivel 2: JOIN + GROUP BY — empresas con más postulaciones ───────────────
# Pregunta de negocio: ¿qué empresa ha recibido más postulaciones?
# Necesitan JOIN entre postulaciones y ofertas, luego GROUP BY empresa.

query_19 = """
SELECT 
    o.empresa                           AS empresa,
    COUNT(*) AS total_postulaciones,
    ROUND(AVG(p.dias_hasta_respuesta), 1) AS dias_promedio_respuesta
FROM postulaciones p
INNER JOIN ofertas o ON p.oferta_id = o.oferta_id
GROUP BY o.empresa
ORDER BY total_postulaciones DESC
LIMIT 10
"""

ejecutar_sql(query_19)


,empresa,total_postulaciones,dias_promedio_respuesta
0,Bancolombia Digital,148,22.6
1,Truora,141,22.0
2,Habi,116,22.2
3,Nequi,115,24.7
4,Pragma,113,22.0
5,Siigo,110,22.8
6,TechCorp Colombia,109,23.4
7,Platzi,106,23.2
8,MercadoLibre,106,21.6
9,Bold,104,23.3


In [46]:
# ─── Nivel 3: Desde cero — el query final de Camila ─────────────────────────
# '¿Cuál es el área tech donde los candidatos CONTRATADOS
#  tardaron en promedio más días en recibir respuesta?'
# Pista: estado_postulacion = 'Contratada'
# Necesitan JOIN (mínimo 2 tablas) + GROUP BY + ORDER BY

query_20 = """
SELECT
    c.area_interes      AS "Nombre de el Área",
    ROUND(AVG(p.dias_hasta_respuesta), 1)   AS "Días_Promedio_Respuesta"

FROM postulaciones p
INNER JOIN candidatos c ON c.candidato_id = p.candidato_id
WHERE p.estado_postulacion = 'Contratada'
GROUP BY c.area_interes
ORDER BY Días_Promedio_Respuesta DESC

"""

ejecutar_sql(query_20)


,Nombre de el Área,Días_Promedio_Respuesta
0,Frontend,26.7
1,Diseño UX/UI,26.5
2,BI & Analytics,26.0
3,Data Science,25.9
4,Full Stack,25.7
5,Backend,25.7
6,Ciberseguridad,25.1
7,Mobile,24.4
8,DevOps,24.3
9,QA,22.2


---

Este es el anticipo de Clase 4. Los datos que extrajeron con SQL ya pueden alimentar un modelo.

In [47]:
# ─── Pipeline completo: SQL → DataFrame listo para modelo ───────────────────
# pd.read_sql_query() ejecuta SQL directamente y devuelve un DataFrame.
# Ese DataFrame puede entrar a un pipeline de preprocesamiento y luego a un modelo.

query_final = """
SELECT 
    c.area_interes,
    c.nivel,
    c.anos_experiencia,
    c.salario_esperado_cop,
    o.salario_maximo_cop,
    p.estado_postulacion
FROM postulaciones p
INNER JOIN candidatos c ON p.candidato_id = c.candidato_id
INNER JOIN ofertas    o ON p.oferta_id    = o.oferta_id
WHERE p.estado_postulacion IN ('Contratada', 'Rechazada')
"""

df_para_modelo = pd.read_sql_query(query_final, conn)

print(f'Shape del dataset: {df_para_modelo.shape}')
print('\nPrimeras filas:')
df_para_modelo.head()


Shape del dataset: (714, 6)

Primeras filas:


,area_interes,nivel,anos_experiencia,salario_esperado_cop,salario_maximo_cop,estado_postulacion
0,Backend,Junior,0,2178340,13978440,Contratada
1,Diseño UX/UI,Junior,1,3498028,9570594,Rechazada
2,Machine Learning,Lead,14,17296903,3998412,Rechazada
3,QA,Lead,10,13656394,12640316,Contratada
4,Backend,Semi-Senior,4,6171581,6406290,Contratada
